In [2]:
!pip install yt-dlp --quiet

In [34]:
import yt_dlp
import os

BASE_DIR = "D:/UGC_Project/raw_videos"
YOUTUBE_DIR = os.path.join(BASE_DIR, "youtube")
os.makedirs(YOUTUBE_DIR, exist_ok=True)

CHANNELS = [
    "https://www.youtube.com/@PhysicsWallah/shorts",   # Physics Wallah - Alakh Pandey — confirmed working Shorts tab, teacher/educator, solo direct-to-camera
]
VIDEOS_PER_CHANNEL = 40
MIN_DURATION = 5
MAX_DURATION = 60  # covers both short ads and longer natural vlog clips



SKIP_KEYWORDS = ["compilation", "full episode", "podcast", "trailer", "teaser", "movie"]

def is_useful_title(title):
    title_lower = title.lower()
    return not any(bad_word in title_lower for bad_word in SKIP_KEYWORDS)

In [35]:
def download_channel_videos(channel_url, limit=40):
    ydl_opts = {
        'format': 'bestvideo[height<=720][ext=mp4]+bestaudio[ext=m4a]/best[height<=720]/best',
        'outtmpl': os.path.join(YOUTUBE_DIR, '%(uploader)s_%(id)s.%(ext)s'),
        'playlistend': limit,
        'ignoreerrors': True,
        'quiet': False,
        'no_warnings': True,
        'merge_output_format': 'mp4',
        'match_filter': yt_dlp.utils.match_filter_func(
            f"duration >= {MIN_DURATION} & duration <= {MAX_DURATION}"
        ),
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([channel_url])

In [36]:
for channel in CHANNELS:
    print(f"\n{'='*50}\nDownloading from: {channel}\n{'='*50}")
    try:
        download_channel_videos(channel, limit=VIDEOS_PER_CHANNEL)
    except Exception as e:
        print(f"Failed on {channel}: {e}")

print(f"\nTotal files in folder: {len(os.listdir(YOUTUBE_DIR))}")


[youtube:tab] Extracting URL: https://www.youtube.com/@PhysicsWallah/shorts
[youtube:tab] @PhysicsWallah/shorts: Downloading webpage
[download] Downloading playlist: Physics Wallah - Alakh Pandey - Shorts
[youtube:tab] Playlist Physics Wallah - Alakh Pandey - Shorts: Downloading 40 items
[download] Downloading item 1 of 40
[youtube] Extracting URL: https://www.youtube.com/shorts/-g8YgOd6QnM
[youtube] -g8YgOd6QnM: Downloading webpage
[youtube] -g8YgOd6QnM: Downloading visionos player API JSON
[youtube] -g8YgOd6QnM: Downloading m3u8 information
[info] -g8YgOd6QnM: Downloading 1 format(s): 396+140
[download] Destination: D:\UGC_Project\raw_videos\youtube\Physics Wallah - Alakh Pandey_-g8YgOd6QnM.f396.mp4
[download] 100% of    2.03MiB in 00:00:01 at 1.02MiB/s   
[download] Destination: D:\UGC_Project\raw_videos\youtube\Physics Wallah - Alakh Pandey_-g8YgOd6QnM.f140.m4a
[download] 100% of  923.01KiB in 00:00:00 at 1.23MiB/s   
[Merger] Merging formats into "D:\UGC_Project\raw_videos\youtub

### 1) wow cosmatics 
### 2) mamearth
### 3) josh talks 
### 4) ashish chanchalani 
### 5) culfit
### 6) physics wallah 
### 7) trakin Tech
### 8) sugar cosmatics 
### 9) technical guruaji
### 10) macaffiene 
### 11) dhusyant kukreja
### 12) priyal kukreja
### 13) flying beast 
### 14) dhruv rathi 
### 15) 1 video from raj shamani 


In [37]:
!pip install requests tqdm --quiet

In [2]:
# ============================================
# CELL 2: Setup
# ============================================
import requests
import os
import time
from tqdm import tqdm
 
PIXABAY_API_KEY = "57245838-b481d2cda5201b8b1342b4c65"
 
BASE_DIR = "D:/UGC_Project/pixabay"
PIXABAY_GLOBAL_DIR = os.path.join(BASE_DIR, "pixabay_global")
os.makedirs(PIXABAY_GLOBAL_DIR, exist_ok=True)
 
TARGET_COUNT = 350
 
# General/global queries - no region restriction, broad talking + movement + expression coverage
QUERIES = [
    "woman talking camera",
    "man talking camera",
    "person explaining product",
    "influencer talking selfie video",
    "person walking outdoor",
    "person running park",
    "person smiling camera",
    "person laughing natural",
    "hand gestures talking",
    "casual conversation video",
    "person sitting talking",
    "vlogger talking head",
    "street interview people",
    "person using phone walking",
    "natural body language video",
]
 
PER_QUERY_LIMIT = 40
 

In [3]:
# ============================================
# CELL 3: Filtering + download helpers
# ============================================
def is_relevant(video):
    if video.get("type") == "animation":
        return False
    relevant_keywords = ["talking", "woman", "man", "person", "influencer", "vlog",
                          "interview", "walking", "running", "smiling", "laughing", "gesture"]
    tags = video.get("tags", "").lower()
    return any(kw in tags for kw in relevant_keywords)
 
def download_file(url, filepath):
    if os.path.exists(filepath):
        return True
    try:
        r = requests.get(url, stream=True, timeout=30)
        r.raise_for_status()
        with open(filepath, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        return True
    except Exception as e:
        print(f"Failed: {url} -> {e}")
        return False
 

In [4]:
# ============================================
# CELL 4: Pixabay search + collect (resume-aware)
# ============================================
def search_pixabay(query, per_page=40):
    url = "https://pixabay.com/api/videos/"
    params = {"key": PIXABAY_API_KEY, "q": query, "per_page": per_page}
    r = requests.get(url, params=params, timeout=20)
    r.raise_for_status()
    return r.json().get("hits", [])

def collect_pixabay_global(target_count=350):
    existing = len(os.listdir(PIXABAY_GLOBAL_DIR))
    print(f"Already have {existing} files — need {target_count - existing} more\n")

    total_skipped = 0

    for query in QUERIES:
        current_count = len(os.listdir(PIXABAY_GLOBAL_DIR))
        if current_count >= target_count:
            print(f"\nTarget of {target_count} reached — stopping.")
            break

        print(f"\nSearching Pixabay: '{query}'")
        try:
            hits = search_pixabay(query, per_page=PER_QUERY_LIMIT)
        except Exception as e:
            print(f"Query failed: {e}")
            continue

        for v in tqdm(hits, desc=query[:20]):
            if len(os.listdir(PIXABAY_GLOBAL_DIR)) >= target_count:
                break
            if not is_relevant(v):
                total_skipped += 1
                continue
            video_url = v["videos"]["medium"]["url"]
            filename = f"pixabay_global_{v['id']}.mp4"
            filepath = os.path.join(PIXABAY_GLOBAL_DIR, filename)
            download_file(video_url, filepath)
            time.sleep(0.3)

    final = len(os.listdir(PIXABAY_GLOBAL_DIR))
    print(f"\nTotal skipped (irrelevant/animation): {total_skipped}")
    print(f"Files in folder: {final}")

collect_pixabay_global(target_count=TARGET_COUNT)

Already have 180 files — need 170 more


Searching Pixabay: 'woman talking camera'


woman talking camera: 100%|████████████████████████████████████████████████████████████| 40/40 [00:05<00:00,  6.76it/s]



Searching Pixabay: 'man talking camera'


man talking camera: 100%|██████████████████████████████████████████████████████████████| 40/40 [00:05<00:00,  7.22it/s]



Searching Pixabay: 'person explaining product'


person explaining pr: 100%|████████████████████████████████████████████████████████████| 40/40 [00:09<00:00,  4.08it/s]


... (21 outputs left out)

person using phone w: 100%|████████████████████████████████████████████████████████████| 40/40 [00:08<00:00,  4.46it/s]



Searching Pixabay: 'natural body language video'


natural body languag: 100%|████████████████████████████████████████████████████████████| 40/40 [00:01<00:00, 21.25it/s]


Total skipped (irrelevant/animation): 286
Files in folder: 183


In [9]:
# ============================================
# CELL 2: Setup (updated queries)
# ============================================
import requests
import os
import time
from tqdm import tqdm

PIXABAY_API_KEY = "57245838-b481d2cda5201b8b1342b4c65"

BASE_DIR = "D:/UGC_Project/pixabay"
PIXABAY_GLOBAL_DIR = os.path.join(BASE_DIR, "pixabay_global")
os.makedirs(PIXABAY_GLOBAL_DIR, exist_ok=True)
TARGET_COUNT = 350

QUERIES = [
    "woman talking camera",
    "man talking camera",
    "person explaining product",
    "influencer talking selfie video",
    "person walking outdoor",
    "person running park",
    "person smiling camera",
    "person laughing natural",
    "hand gestures talking",
    "casual conversation video",
    "person sitting talking",
    "vlogger talking head",
    "street interview people",
    "person using phone walking",
    "natural body language video",
    # new queries for variety
    "person waving hello",
    "friends talking outdoor",
    "podcast interview studio",
    "person shopping mall",
    "couple walking street",
    "office employee talking",
    "person cooking kitchen",
    "student talking campus",
    "person nodding head",
    "customer product review",
]

PER_QUERY_LIMIT = 40
MAX_PAGES_PER_QUERY = 5   # fetch up to 5 pages (200 results) per query before moving on

In [10]:
# ============================================
# CELL 4: Pixabay search + collect (paginated, resume-aware)
# ============================================
def search_pixabay(query, page=1, per_page=40):
    url = "https://pixabay.com/api/videos/"
    params = {"key": PIXABAY_API_KEY, "q": query, "page": page, "per_page": per_page}
    r = requests.get(url, params=params, timeout=20)
    r.raise_for_status()
    return r.json().get("hits", [])

def collect_pixabay_global(target_count=350):
    existing = len(os.listdir(PIXABAY_GLOBAL_DIR))
    print(f"Already have {existing} files — need {target_count - existing} more\n")

    total_skipped = 0

    for query in QUERIES:
        if len(os.listdir(PIXABAY_GLOBAL_DIR)) >= target_count:
            print(f"\nTarget of {target_count} reached — stopping.")
            break

        for page in range(1, MAX_PAGES_PER_QUERY + 1):
            if len(os.listdir(PIXABAY_GLOBAL_DIR)) >= target_count:
                break

            print(f"\nSearching Pixabay: '{query}' (page {page})")
            try:
                hits = search_pixabay(query, page=page, per_page=PER_QUERY_LIMIT)
            except Exception as e:
                print(f"Query failed: {e}")
                break

            if not hits:
                print("No more results for this query — moving on.")
                break

            new_this_page = 0
            for v in tqdm(hits, desc=f"{query[:15]} p{page}"):
                if len(os.listdir(PIXABAY_GLOBAL_DIR)) >= target_count:
                    break
                if not is_relevant(v):
                    total_skipped += 1
                    continue
                video_url = v["videos"]["medium"]["url"]
                filename = f"pixabay_global_{v['id']}.mp4"
                filepath = os.path.join(PIXABAY_GLOBAL_DIR, filename)
                if not os.path.exists(filepath):
                    if download_file(video_url, filepath):
                        new_this_page += 1
                time.sleep(0.3)

            # if a full page yields nothing new, stop paginating this query
            if new_this_page == 0:
                break

    final = len(os.listdir(PIXABAY_GLOBAL_DIR))
    print(f"\nTotal skipped (irrelevant/animation): {total_skipped}")
    print(f"Files in folder: {final}")

collect_pixabay_global(target_count=TARGET_COUNT)

Already have 183 files — need 167 more


Searching Pixabay: 'woman talking camera' (page 1)


woman talking c p1: 100%|██████████████████████████████████████████████████████████████| 40/40 [00:05<00:00,  6.83it/s]



Searching Pixabay: 'man talking camera' (page 1)


man talking cam p1: 100%|██████████████████████████████████████████████████████████████| 40/40 [00:05<00:00,  7.20it/s]



Searching Pixabay: 'person explaining product' (page 1)


person explaini p1: 100%|██████████████████████████████████████████████████████████████| 40/40 [00:09<00:00,  4.08it/s]


... (67 outputs left out)

couple walking  p2: 100%|██████████████████████████████████████████████████████████████| 40/40 [02:19<00:00,  3.50s/it]



Searching Pixabay: 'couple walking street' (page 3)


couple walking  p3: 100%|██████████████████████████████████████████████████████████████| 40/40 [02:35<00:00,  3.88s/it]


Target of 350 reached — stopping.

Total skipped (irrelevant/animation): 852
Files in folder: 350
